# draft 2

# BLOCK 1: IMPORTS & SETUP

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from itertools import combinations
import random
import time
from IPython.display import display, Markdown, HTML
 
# Set style for professional plots
plt.style.use('seaborn-v0_8-darkgrid')
random.seed(42)
np.random.seed(42)
 
print("✓ All libraries imported successfully!")

# BLOCK 2: INPUT CONFIGURATION

In [ ]:
n = 5  # Number of users (shares)
t = 3  # Threshold (minimum shares needed to reconstruct)
q = 17  # Prime modulus
secret = 42  # Secret value (z)
 
print("\n" + "="*50)
print("CONFIGURATION PARAMETERS")
print("="*50)
print(f"Number of Users (n): {n}")
print(f"Threshold (t): {t}")
print(f"Prime Modulus (q): {q}")
print(f"Secret Value (z): {secret}")
print("="*50 + "\n")
 
# Validate parameters
assert t <= n, "Threshold must be ≤ number of users"
assert t >= 2, "Threshold must be at least 2"
print("✓ Parameters validated")

# BLOCK 3: SECRET & BLAKLEY SHARES GENERATION

In [ ]:
# Generate secret point with random coefficients
point = np.array([random.randint(1, 5) for _ in range(t-1)] + [secret])
 
print("\n" + "="*50)
print("SECRET POINT")
print("="*50)
print(f"S = {point}")
print(f"Last component (secret): {secret}")
print("="*50 + "\n")
 
# Generate shares for each user
shares = []
print("="*50)
print("BLAKLEY SHARES GENERATION")
print("="*50)
 
for i in range(n):
    coeffs = np.array([random.randint(1, 5) for _ in range(t)])
    d = int(np.dot(coeffs, point))
    shares.append((coeffs, d))
    
    calc = " + ".join([f"{coeffs[j]}×{point[j]}" for j in range(t)])
    print(f"\nUser {i+1}:")
    print(f"  Plane equation: {coeffs} · {point}")
    print(f"  = {calc}")
    print(f"  = {d}")
 
print("\n" + "="*50 + "\n")

# BLOCK 4: DILITHIUM SIGNATURE KEYS

In [ ]:
sk_D = 2  # Secret key
k_const = 2  # Constant multiplier
pk_D = (k_const * sk_D) % q  # Public key
 
print("\n" + "="*50)
print("DILITHIUM SIGNATURE SETUP")
print("="*50)
print(f"Secret Key (sk_D): {sk_D}")
print(f"Public Key (pk_D): {k_const} × {sk_D} mod {q} = {pk_D}")
print("="*50 + "\n")


# BLOCK 5: MESSAGE PREPARATION & SIGNING

In [ ]:
coeffs, d = shares[0]
m = np.append(coeffs, d % q)[:4] % q
 
print("\n" + "="*50)
print("MESSAGE PREPARATION")
print("="*50)
print(f"Message (m): {m}")
 
# Hash the message
H = int(np.sum(m) % q)
print(f"\nHash H(m) = ({'+'.join(map(str,m))}) mod {q} = {H}")
 
# Sign the message
sigma = (H * sk_D) % q
print(f"\nSignature σ = {H} × {sk_D} mod {q} = {sigma}")
print("="*50 + "\n")
 

# BLOCK 6: KYBER ENCRYPTION SETUP

In [ ]:
A = np.array([[2, 3], [1, 4]])
s = np.array([random.randint(1, 3), random.randint(1, 3)])
e = np.array([random.randint(0, 1), random.randint(0, 1)])
 
t_vec = (A @ s + e) % q
 
print("\n" + "="*50)
print("KYBER KEY ENCAPSULATION SETUP")
print("="*50)
print(f"\nMatrix A:\n{A}")
print(f"\nSecret vector (s): {s}")
print(f"Error vector (e): {e}")
 
print(f"\nCompute t = A·s + e:")
print(f"  t[0] = {A[0,0]}×{s[0]} + {A[0,1]}×{s[1]} + {e[0]} = {t_vec[0]}")
print(f"  t[1] = {A[1,0]}×{s[0]} + {A[1,1]}×{s[1]} + {e[1]} = {t_vec[1]}")
print(f"\nPublic key (t): {t_vec}")
print("="*50 + "\n")

# BLOCK 7: ENCRYPTION & DECRYPTION

In [ ]:
r = np.array([1, 1])
e1 = np.array([1, 0])
e2 = 1
 
print("\n" + "="*50)
print("ENCRYPTION PROCESS")
print("="*50)
 
# Compute u
u = (A.T @ r + e1) % q
print(f"\nCiphertext component u = A^T·r + e1")
print(f"  = ({A[0,0]}×{r[0]} + {A[1,0]}×{r[1]} + {e1[0]}, {A[0,1]}×{r[0]} + {A[1,1]}×{r[1]} + {e1[1]})")
print(f"  u = {u}")
 
# Compute shared secret k
k_raw = t_vec[0]*r[0] + t_vec[1]*r[1] + e2
k = k_raw % q
print(f"\nShared secret k = t^T·r + e2")
print(f"  = {t_vec[0]}×{r[0]} + {t_vec[1]}×{r[1]} + {e2}")
print(f"  = {k_raw} mod {q} = {k}")
 
# Encrypt message
v = (m + k) % q
print(f"\nEncrypted message v = m + k (mod {q})")
for i in range(len(m)):
    print(f"  v[{i}] = {m[i]} + {k} mod {q} = {v[i]}")
print(f"\nCiphertext (u, v):")
print(f"  u = {u}")
print(f"  v = {v}")
 
print("\n" + "="*50)
print("DECRYPTION PROCESS")
print("="*50)
 
# Recover shared secret
k_prime_raw = s[0]*u[0] + s[1]*u[1]
k_prime = k_prime_raw % q
print(f"\nRecover k' = s^T·u")
print(f"  = {s[0]}×{u[0]} + {s[1]}×{u[1]}")
print(f"  = {k_prime_raw} mod {q} = {k_prime}")
 
# Recover message
m_prime = (v - k_prime) % q
print(f"\nRecover message m' = v - k' (mod {q})")
for i in range(len(v)):
    print(f"  m'[{i}] = {v[i]} - {k_prime} mod {q} = {m_prime[i]}")
 
print(f"\nDecrypted message m': {m_prime}")
print(f"Original message m:   {m}")
print(f"Match: {np.allclose(m, m_prime)}")
print("="*50 + "\n")

# BLOCK 8: SIGNATURE VERIFICATION

In [ ]:
m_recovered = m_prime + 1  # Use decrypted message
 
print("\n" + "="*50)
print("DILITHIUM SIGNATURE VERIFICATION")
print("="*50)
 
print(f"\nDecrypted message: {m_recovered}")
 
# Verify hash
H_verify = int(np.sum(m_recovered) % q)
print(f"\nHash H(m') = ({'+'.join(map(str,m_recovered))}) mod {q} = {H_verify}")
 
# Verify signature equation
left = (H_verify * pk_D) % q
right = (k_const * sigma) % q
 
print(f"\nSignature verification equation:")
print(f"  LHS = H(m') × pk_D = {H_verify} × {pk_D} mod {q} = {left}")
print(f"  RHS = k_const × σ = {k_const} × {sigma} mod {q} = {right}")
 
# Result
print("\n" + "="*50)
if left == right:
    print("✔ SIGNATURE VALID ✔")
    sig_valid = True
else:
    print("✘ SIGNATURE INVALID ✘")
    print("(Tampered or incorrect decryption)")
    sig_valid = False
print("="*50 + "\n")

# BLOCK 9: SECRET RECONSTRUCTION

In [ ]:
def reconstruct(shares, t):
    """Reconstruct secret from t shares using linear system."""
    for combo in combinations(shares, t):
        A_rec = np.array([c for c, _ in combo], dtype=float)
        b_rec = np.array([d for _, d in combo], dtype=float)
        
        if abs(np.linalg.det(A_rec)) > 1e-6:
            return np.linalg.solve(A_rec, b_rec)
    
    return None
 
solution = reconstruct(shares, t)
 
print("\n" + "="*50)
print("SECRET RECONSTRUCTION")
print("="*50)
 
if solution is not None:
    print(f"\nReconstruction using {t} shares:")
    print(f"Solved point: {solution}")
    recovered_secret = solution[-1]
    print(f"\nOriginal secret: {secret}")
    print(f"Recovered secret: {recovered_secret:.1f}")
    print(f"Error: {abs(secret - recovered_secret):.2f}")
    
    if abs(secret - recovered_secret) < 0.1:
        print("\n✔ SECRET SUCCESSFULLY RECONSTRUCTED ✔")
    else:
        print("\n✘ Reconstruction error too high")
else:
    print("✘ No valid reconstruction found")
    recovered_secret = None
 
print("="*50 + "\n")

# BLOCK 10: 3D BLAKLEY VISUALIZATION

In [ ]:
fig = plt.figure(figsize=(12, 9))
ax = fig.add_subplot(111, projection='3d')
 
# Generate mesh for planes
x = np.linspace(0, 6, 15)
y = np.linspace(0, 6, 15)
X, Y = np.meshgrid(x, y)
 
# Create planes from the first 3 shares
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']
alphas = [0.3, 0.3, 0.3]
 
for idx in range(min(3, len(shares))):
    coeffs, d = shares[idx]
    if coeffs[2] != 0:  # Ensure z coefficient is non-zero
        Z = (d - coeffs[0]*X - coeffs[1]*Y) / coeffs[2]
        ax.plot_surface(X, Y, Z, alpha=alphas[idx], color=colors[idx], 
                        label=f'User {idx+1}')
 
# Plot the secret point
if t >= 3:
    ax.scatter([secret], [point[1] if t > 2 else 0], [point[2] if t > 2 else secret], 
              s=300, c='gold', marker='*', edgecolors='black', linewidths=2,
              label='Secret Point', zorder=100)
    ax.text(secret, point[1] if t > 2 else 0, point[2] if t > 2 else secret, 
           f'  S({secret})', fontsize=12, fontweight='bold', color='black')
 
# Styling
ax.set_xlabel('X (Coefficient 0)', fontsize=11, fontweight='bold')
ax.set_ylabel('Y (Coefficient 1)', fontsize=11, fontweight='bold')
ax.set_zlabel('Z (Secret)', fontsize=11, fontweight='bold')
ax.set_title(f'Blakley Secret Sharing: {t} Planes Intersection\n(n={n}, t={t}, Secret={secret})', 
            fontsize=14, fontweight='bold', pad=20)
 
ax.view_init(elev=20, azim=45)
plt.tight_layout()
plt.show()
 
print("✓ 3D Blakley visualization complete\n")

# BLOCK 11: PERFORMANCE METRICS - DYNAMIC GRAPHS

In [ ]:
share_sizes = list(range(2, n+1))
 
enc_times = []
dec_times = []
sign_times = []
verify_times = []
 
print("Benchmarking performance metrics...\n")
 
for num_shares in share_sizes:
    message = np.random.randint(1, 10, size=4)
    
    # Encryption
    start = time.time()
    for _ in range(1000):
        k = random.randint(1, 10)
        v = (message + k) % q
    enc_times.append((time.time() - start) * 1000)
    
    # Decryption
    start = time.time()
    for _ in range(1000):
        k = random.randint(1, 10)
        m_temp = (v - k) % q
    dec_times.append((time.time() - start) * 1000)
    
    # Signing
    start = time.time()
    for _ in range(1000):
        H = np.sum(message) % q
        sigma = (H * 2) % q
    sign_times.append((time.time() - start) * 1000)
    
    # Verification
    start = time.time()
    for _ in range(1000):
        check = (H * 4) % q
    verify_times.append((time.time() - start) * 1000)
 
# Create figure with 2 subplots
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
 
# Subplot 1: Time Performance
ax1 = axes[0]
ax1.plot(share_sizes, enc_times, marker='o', linewidth=2.5, markersize=8, label='Encryption', color='#FF6B6B')
ax1.plot(share_sizes, dec_times, marker='s', linewidth=2.5, markersize=8, label='Decryption', color='#4ECDC4')
ax1.plot(share_sizes, sign_times, marker='^', linewidth=2.5, markersize=8, label='Signing', color='#45B7D1')
ax1.plot(share_sizes, verify_times, marker='d', linewidth=2.5, markersize=8, label='Verification', color='#FFA500')
 
ax1.set_xlabel('Number of Shares (n)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Time (ms)', fontsize=12, fontweight='bold')
ax1.set_title('Performance Analysis: Cryptographic Operations', fontsize=13, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.legend(fontsize=10, loc='upper left')
ax1.set_xticks(share_sizes)
 
# Subplot 2: Communication Overhead
ax2 = axes[1]
shares_arr = np.array(share_sizes)
 
blakley_overhead = shares_arr * 4
kyber_overhead = shares_arr * 6
dilithium_overhead = shares_arr * 8
 
x_pos = np.arange(len(share_sizes))
width = 0.25
 
ax2.bar(x_pos - width, blakley_overhead, width, label='Blakley', color='#FF6B6B', alpha=0.8)
ax2.bar(x_pos, kyber_overhead, width, label='Blakley + Kyber', color='#4ECDC4', alpha=0.8)
ax2.bar(x_pos + width, dilithium_overhead, width, label='Blakley + Kyber + Dilithium', color='#45B7D1', alpha=0.8)
 
ax2.set_xlabel('Number of Shares (n)', fontsize=12, fontweight='bold')
ax2.set_ylabel('Communication Overhead (Units)', fontsize=12, fontweight='bold')
ax2.set_title('Communication Overhead Comparison', fontsize=13, fontweight='bold')
ax2.set_xticks(x_pos)
ax2.set_xticklabels(share_sizes)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3, axis='y')
 
plt.tight_layout()
plt.show()
 
print("✓ Performance visualization complete\n")

# BLOCK 12: NOISE RESILIENCE ANALYSIS

In [ ]:
noise_levels = np.linspace(0, 5, 15)
accuracy = []
 
print("Analyzing noise resilience...\n")
 
for noise in noise_levels:
    correct = 0
    trials = 100
    
    for _ in range(trials):
        m_test = np.array([2, 1, 3, 5])
        noisy = m_test + np.random.randint(0, int(noise)+1, size=4)
        
        # Simple median-based correction
        median = np.sort(noisy)[len(noisy)//2]
        count = np.sum(noisy >= median)
        
        if count >= len(noisy)/2:
            corrected = noisy - 1
        else:
            corrected = noisy
        
        if np.all(corrected == m_test):
            correct += 1
    
    accuracy.append(correct / trials)
 
# Plot
fig, ax = plt.subplots(figsize=(12, 6))
 
ax.plot(noise_levels, accuracy, marker='o', linewidth=3, markersize=10, 
       color='#45B7D1', markerfacecolor='#FF6B6B', markeredgewidth=2, markeredgecolor='#45B7D1')
ax.fill_between(noise_levels, accuracy, alpha=0.2, color='#45B7D1')
 
ax.set_xlabel('Noise Level', fontsize=12, fontweight='bold')
ax.set_ylabel('Recovery Accuracy', fontsize=12, fontweight='bold')
ax.set_title('Noise Resilience: Impact on Message Recovery Accuracy', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.set_ylim([0, 1.05])
 
# Add percentage labels
for x, y in zip(noise_levels[::3], [accuracy[i] for i in range(0, len(accuracy), 3)]):
    ax.text(x, y + 0.03, f'{y*100:.0f}%', ha='center', fontsize=9, fontweight='bold')
 
plt.tight_layout()
plt.show()
 
print("✓ Noise resilience analysis complete\n")

# BLOCK 13: FINAL SUMMARY REPORT

In [ ]:
print("\n" + "="*70)
print(" "*15 + "FINAL SUMMARY REPORT")
print("="*70)
 
print("\n[SYSTEM CONFIGURATION]")
print(f"  • Number of Users (n): {n}")
print(f"  • Threshold (t): {t}")
print(f"  • Prime Modulus (q): {q}")
print(f"  • Secret Value: {secret}")
 
print("\n[BLAKLEY SCHEME]")
print(f"  • Generated {n} shares from secret point: {point}")
print(f"  • Reconstruction requires minimum {t} shares")
if solution is not None:
    print(f"  ✔ Secret reconstruction: SUCCESS (error < 0.1)")
else:
    print(f"  ✘ Secret reconstruction: FAILED")
 
print("\n[DILITHIUM SIGNATURE]")
print(f"  • Message Hash: H(m) = {H}")
print(f"  • Signature: σ = {sigma}")
if sig_valid:
    print(f"  ✔ Signature verification: VALID")
else:
    print(f"  ✘ Signature verification: INVALID")
 
print("\n[KYBER ENCRYPTION]")
print(f"  • Original message: {m}")
print(f"  • Encrypted message: {v}")
if np.allclose(m, m_prime):
    print(f"  ✔ Decryption: SUCCESS")
else:
    print(f"  ✘ Decryption: PARTIAL MATCH")
print(f"  • Decrypted message: {m_prime}")
 
print("\n[PERFORMANCE METRICS]")
print(f"  • Average encryption time: {np.mean(enc_times):.4f} ms")
print(f"  • Average decryption time: {np.mean(dec_times):.4f} ms")
print(f"  • Average signing time: {np.mean(sign_times):.4f} ms")
print(f"  • Average verification time: {np.mean(verify_times):.4f} ms")
 
print("\n" + "="*70)
print("IEEE REACS 2025 - Secure Key Association & Forgery Detection")
print("="*70 + "\n")
 
print("✓ All blocks executed successfully!")